# 🐄 BovWeight CR — Lanzador del Microservicio

Este notebook clona el repositorio, instala todas las dependencias (YOLOv8 + Depth Pro) y expone el microservicio FastAPI mediante ngrok.

In [ ]:
import os
from google.colab import userdata
GITHUB_REPO = "https://github.com/Mayloparra24/bovweight-api.git"
BRANCH = "main"

# Retrieve the token and strip any extraneous quotes
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", userdata.get('Token')).strip('"\'')

print(f"Repo: {GITHUB_REPO}")
print(f"Branch: {BRANCH}")

In [ ]:
# 1. Clonar el repositorio del microservicio
!git clone -b {BRANCH} {GITHUB_REPO}
%cd bovweight-api

# 2. Instalar dependencias base
!pip install -r requirements.txt -q

# 3. Instalar YOLO (asegurar ultralytics)
%pip install ultralytics -q

print("\u2705 Dependencias base instaladas.")

In [ ]:
# 4. Clonar e instalar Depth Pro (Apple)
%cd /content
!rm -rf ml-depth-pro
!git clone https://github.com/apple/ml-depth-pro.git

# 5. Parche: eliminar restriccion numpy<2
!sed -i 's/numpy<2/numpy/g' ml-depth-pro/pyproject.toml

# 6. Instalar Depth Pro en modo editable
%cd ml-depth-pro
%pip install -e . -q

# 7. Descargar pesos preentrenados
!source get_pretrained_models.sh

# 8. Salvavidas: renombrar archivo de pesos si quedo .pt.1
!if [ -f checkpoints/depth_pro.pt.1 ]; then mv -f checkpoints/depth_pro.pt.1 checkpoints/depth_pro.pt; fi
!if [ ! -f checkpoints/depth_pro.pt ]; then echo "❌ Error en descarga de pesos. Revisa checkpoints/"; fi

%cd /content/bovweight-api

import os
if not os.path.isfile("app/best.pt"):
    print("⚠️ ADVERTENCIA: app/best.pt no encontrado. Subilo manualmente.")
else:
    print("✅ app/best.pt encontrado.")

print("✅ Depth Pro instalado correctamente.")

In [ ]:
# 9. Validar importaciones
import torch
from ultralytics import YOLO
print(f"PyTorch {torch.__version__} | CUDA disponible: {torch.cuda.is_available()}")
print("Importaciones correctas.")

In [ ]:
# 10. Configurar ngrok y lanzar FastAPI
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio # Importar asyncio para la gestión directa del bucle
from uvicorn.config import Config # Importar Config para la configuración explícita del servidor

nest_asyncio.apply()

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Iniciar el túnel ngrok
public_url = ngrok.connect(8000)
print(f"🔗 Microservicio expuesto en: {public_url}")
print(f"📮 POST {public_url}/api/v1/predict-weight")
print(f"🔍 Health check: {public_url}/health")

# Configurar el servidor Uvicorn explícitamente en lugar de usar uvicorn.run()
config = Config(
    "app.main:app",
    host="0.0.0.0",
    port=8000,
    reload=False, # True para desarrollo, False para producción
    log_level="info" # Agregado para una mejor información en Colab
)
server = uvicorn.Server(config)

# Obtener el ciclo de eventos actual y crear una tarea para ejecutar el servidor
# Esto permite que el servidor se ejecute en segundo plano sin bloquear la celda inmediatamente,
# y evita el RuntimeError de las llamadas anidadas a asyncio.run().
loop = asyncio.get_event_loop()
task = loop.create_task(server.serve())

print("✅ Servidor FastAPI iniciado en segundo plano.")
# El servidor se ejecutará hasta que el entorno de ejecución de Colab se desconecte o reinicie,
# o hasta que el proceso se detenga explícitamente.
# Nota: Esta celda completará su ejecución, pero el servidor continuará ejecutándose.